# Week 2 · Day 2 — LangChain: Tools, Chains, Memory & Your First Framework Agent

This notebook rebuilds Day 1's raw-Python agent using **LangChain**, then goes further with
LCEL chains, real tool integration (a local JSON "database"), conversational memory, structured
output, and tool-error handling.

**Note on versions:** this notebook was written against `langchain==1.4.0` / `langchain-anthropic==1.7.1`
(current as of testing). LangChain 1.0 replaced the old `create_tool_calling_agent` + `AgentExecutor`
pattern with a new LangGraph-based `create_agent`. Most "Day 2 LangChain" tutorials online (including
the structure of this assignment) still teach the pre-1.0 pattern, so this notebook uses the
**`langchain-classic`** compatibility package to run that exact pattern with `verbose=True` (needed
for the reasoning trace Task 3 asks for), and calls out the modern equivalent in the write-up.
That version churn is itself one of the "leaky abstraction" lessons for today.


In [1]:
# pip install -q langchain langchain-anthropic langchain-classic langchain-core pydantic

import os
# Set your key before running the cells that actually call the model:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-sonnet-4-5", temperature=0)


## Task 1 — LangChain Setup & Core Concepts

### Mapping Day 1's raw Python to LangChain

| Day 1 (raw Python) | LangChain equivalent | What it actually is under the hood |
|---|---|---|
| Your own `call_claude(messages)` wrapper around the Anthropic SDK | `ChatAnthropic` | A `BaseChatModel` subclass — still calls the same `/v1/messages` endpoint, just returns a `AIMessage` object instead of a raw dict |
| Your hand-written `TOOLS = {name: function}` dict + JSON schema you built by hand | `@tool` decorator / `Tool` class | Wraps a Python function, auto-generates the JSON schema from the type hints + docstring, and exposes `.name`, `.description`, `.args_schema` |
| Your `while True:` loop that called the model, parsed `tool_use` blocks, ran the function, and appended `tool_result` back into the message list | `AgentExecutor` (classic) / `create_agent` (modern, LangGraph-based) | Exactly the same loop — model call → detect tool calls → execute → append result → repeat until a plain text answer. LangChain just hides the loop and the message bookkeeping from you |
| Your `history = []` list you manually appended user/assistant turns to | `ConversationBufferMemory` / `RunnableWithMessageHistory` / LangGraph checkpointer | Same list of messages, now managed by a class that injects it into the prompt automatically each turn |
| A single prompt string you formatted with `.format()` | `PromptTemplate` / `ChatPromptTemplate` + **LCEL** `|` pipelines | A composable, typed pipeline object instead of a plain string |

### LCEL basic chain (prompt → response)


In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

basic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical explainer."),
    ("human", "Explain {concept} in exactly two sentences."),
])

# This is the LCEL pipeline: prompt -> llm -> parser
basic_chain = basic_prompt | llm | StrOutputParser()

# Requires ANTHROPIC_API_KEY to actually run:
# response = basic_chain.invoke({"concept": "LangChain's Runnable interface"})
# print(response)


### What is the `|` pipe actually doing?

Every LangChain component (`ChatPromptTemplate`, `ChatAnthropic`, `StrOutputParser`, tools, even
whole agents) implements the same **`Runnable` interface** — `.invoke()`, `.batch()`, `.stream()`,
`.ainvoke()`. The `|` operator is just Python's overloaded `__or__` method, and LangChain uses it to
build a `RunnableSequence`: `a | b` returns a new `Runnable` whose `.invoke(x)` is literally
`b.invoke(a.invoke(x))`. So `prompt | llm | parser` is sugar for "take my input dict, format it into
messages, feed those messages to the model, take the resulting `AIMessage` and pull out `.content`
as a plain string" — no magic, just a linked list of objects with a shared `.invoke()` contract.


## Task 2 — Define & Register Tools

Three tools below, matching Day 1's set where possible, plus one new tool (`lookup_product_price`)
that reads from a real local data source: `data/products.json`, a small JSON "product database".


In [3]:
import json
import random
from langchain_core.tools import tool

DB_PATH = "data/products.json"

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression (e.g. '899 * 1.05' or '1499 - 899').
    Use this whenever the user asks for a sum, difference, percentage, or comparison
    that requires arithmetic. Input must be a valid Python arithmetic expression
    using only numbers and + - * / ( ) . Do not pass words or variable names."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return f"Error: expression contains disallowed characters: {expression}"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error evaluating expression: {e}"


@tool
def word_counter(text: str) -> str:
    """Count the number of words in a piece of text. Use this when the user asks
    how long a piece of text is, or how many words something contains."""
    return str(len(text.split()))


@tool
def lookup_product_price(product_id: str) -> str:
    """Look up a product's live price and stock from the internal product database.
    Input must be a product_id key such as 'laptop_x1', 'laptop_z9', 'phone_alpha',
    'phone_beta', 'headphones_air', or 'headphones_pro'. Returns JSON with name,
    price_usd, category and stock. Use this tool any time the user asks for the
    price of a specific product before doing any comparison or math on prices."""
    with open(DB_PATH) as f:
        db = json.load(f)
    if product_id not in db:
        return f"Error: no product with id '{product_id}'. Known ids: {list(db.keys())}"
    # Simulate an occasionally-flaky external data source, the way a real API would behave
    if random.random() < 0.2:
        raise ConnectionError("Simulated transient API timeout while fetching product data")
    return json.dumps(db[product_id])


tools = [calculator, word_counter, lookup_product_price]
for t in tools:
    print(f"{t.name}: {t.description[:70]}...")


calculator: Evaluate a basic arithmetic expression (e.g. '899 * 1.05' or '1499 - 8...
word_counter: Count the number of words in a piece of text. Use this when the user a...
lookup_product_price: Look up a product's live price and stock from the internal product dat...


### Why the docstring matters

LangChain doesn't just use the docstring as documentation for humans — it's sent to the model
**verbatim as the tool's `description` field** in the tool-use schema, alongside the auto-derived
JSON schema of the function's arguments (from the type hints). The model has no access to your
function's implementation, only its name, docstring, and argument schema. That means the docstring
*is* the interface: if it's vague ("looks up a product"), the model may call the tool with the wrong
kind of input (a product name instead of a product_id) or skip it entirely when it should be used.
Writing the docstring like a mini system prompt for that one tool — what it does, when to use it,
and the exact input format — is effectively prompt engineering, not documentation.


## Task 3 — Build an Agent with `create_tool_calling_agent` / `AgentExecutor`

Using `langchain_classic` for this exact API (LangChain 1.0 moved the default to
`langchain.agents.create_agent`, a LangGraph-based constructor — see the write-up).


In [4]:
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful shopping assistant. Use tools to get real prices "
               "before comparing products or doing math — never guess a price."),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,            # prints the full reason -> act -> observe trace
    handle_parsing_errors=True,
)

# Requires ANTHROPIC_API_KEY to actually run:
# result = agent_executor.invoke({
#     "input": "What's the price difference between laptop_x1 and laptop_z9, "
#              "and what percentage more expensive is the z9?"
# })
# print(result["output"])


### Annotated trace (illustrative — run the cell above with a real key to capture your own)

With `verbose=True`, `AgentExecutor` prints one block per loop iteration. For the multi-step query
above, the shape of the trace looks like this:

```
> Entering new AgentExecutor chain...

[REASON]  Model decides it needs laptop_x1's price before it can compare anything.
[ACT]     Invoking: `lookup_product_price` with `{'product_id': 'laptop_x1'}`
[OBSERVE] {"name": "Laptop X1", "price_usd": 899, "category": "electronics", "stock": 12}

[REASON]  Model now needs laptop_z9's price for the comparison.
[ACT]     Invoking: `lookup_product_price` with `{'product_id': 'laptop_z9'}`
[OBSERVE] {"name": "Laptop Z9 Pro", "price_usd": 1499, "category": "electronics", "stock": 5}

[REASON]  Model has both prices, now needs the percentage difference — hands off to calculator
          instead of doing the arithmetic itself.
[ACT]     Invoking: `calculator` with `{'expression': '(1499 - 899) / 899 * 100'}`
[OBSERVE] 66.74...

[REASON]  Model has everything it needs, composes final answer.
> Finished chain.
```

### Comparing this to Day 1's raw log

**Similar:** the underlying loop is identical — reason (model picks a tool), act (call the
function), observe (feed the result back in), repeat until the model stops calling tools. Every
`[ACT]`/`[OBSERVE]` pair here is exactly one iteration of Day 1's `while True` loop.

**What's hidden now:**
- The raw `tool_use` / `tool_result` message blocks and their IDs — `AgentExecutor` manages the
  message list internally; you only see `.invoke()` and the printed trace, not the actual API
  payloads going back and forth.
- The stopping condition — Day 1 you wrote the `if response.stop_reason == "end_turn": break`
  check yourself; here it's baked into `AgentExecutor`'s loop and only exposed via
  `max_iterations` / `max_execution_time` kwargs if you want to override it.
- Parsing of the model's tool-call format — Day 1 you parsed `content` blocks by hand; LangChain's
  output parser does this and will silently retry/reformat on a malformed call if
  `handle_parsing_errors=True`, which can mask a genuinely broken tool call as if it were the
  model's fault.


## Task 4 — Add Memory

Wrapping the `AgentExecutor` in `RunnableWithMessageHistory` so a 3-turn conversation carries
context (price of X → compare to Y → recommend for a budget-conscious client).


In [5]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# In-memory store keyed by session_id (swap for a DB-backed store in production)
_session_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in _session_store:
        _session_store[session_id] = InMemoryChatMessageHistory()
    return _session_store[session_id]

agent_with_memory = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

config = {"configurable": {"session_id": "day2-demo"}}

# Requires ANTHROPIC_API_KEY to actually run:
# turn1 = agent_with_memory.invoke({"input": "Find the price of laptop_x1"}, config=config)
# turn2 = agent_with_memory.invoke({"input": "Now compare it to laptop_z9"}, config=config)
# turn3 = agent_with_memory.invoke(
#     {"input": "Which one should I recommend to a budget-conscious client?"}, config=config
# )
# print(turn3["output"])


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


**Note:** `RunnableWithMessageHistory` prints a `LangChainDeprecationWarning` in 1.x — LangChain's
own docs now point toward LangGraph's built-in checkpointer/persistence for memory instead. It still
works and is the more approachable API for a first pass at memory, which is why it's used here; the
write-up covers the tradeoff.

For turn 3 to be answered correctly without re-stating the prices, the model has to look back at
`chat_history` (which now contains both the tool calls *and* their results from turns 1–2) rather
than calling `lookup_product_price` a third time — that's the actual behavior worth checking when
you run this for real: does it reuse memory, or re-call the tool unnecessarily?


## Task 5 — Structured Output & Error Handling

### Structured final answer via a Pydantic model


In [6]:
from pydantic import BaseModel, Field

class ProductRecommendation(BaseModel):
    recommended_product: str = Field(description="Name of the recommended product")
    price_usd: float = Field(description="Price of the recommended product in USD")
    reason: str = Field(description="Short justification for the recommendation")
    budget_conscious: bool = Field(description="Whether this pick suits a budget-conscious buyer")

structured_llm = llm.with_structured_output(ProductRecommendation)

structuring_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given the shopping-agent's final answer text below, extract it into the "
               "required structured schema."),
    ("human", "{agent_final_answer}"),
])
structuring_chain = structuring_prompt | structured_llm

# Requires ANTHROPIC_API_KEY to actually run — feed it the free-text answer from Task 4's turn3:
# structured_result = structuring_chain.invoke({"agent_final_answer": turn3["output"]})
# print(structured_result)
# ProductRecommendation(recommended_product='Laptop X1', price_usd=899.0,
#                        reason='Lower price for similar core specs', budget_conscious=True)


**Design choice:** rather than trying to force `AgentExecutor` itself to emit a Pydantic object
(the classic agent's tool-calling loop and structured final-answer output don't compose cleanly —
this is one of the leaky spots), a second small LCEL chain with `with_structured_output` is used to
post-process the agent's free-text final answer into the schema. This is a common pattern: **agent
for reasoning/tool use, a plain structured chain for shaping the final output.**

### Error handling for a flaky tool

`lookup_product_price` above randomly raises `ConnectionError` ~20% of the time, simulating a real
external API being flaky. Two levels of handling:


In [7]:
from langchain_core.tools import ToolException

@tool
def lookup_product_price_safe(product_id: str) -> str:
    """Same as lookup_product_price, but retries once on a transient failure before
    surfacing an error to the agent. Input must be a product_id key."""
    for attempt in range(2):
        try:
            return lookup_product_price.func(product_id)
        except ConnectionError as e:
            if attempt == 0:
                continue  # one retry
            # Give up and hand a clean, model-readable error back instead of a raw traceback
            return f"Error: could not fetch data for '{product_id}' after retrying ({e})"

# AgentExecutor-level handling: if a tool still raises, don't crash the whole run
agent_executor_resilient = AgentExecutor(
    agent=create_tool_calling_agent(llm, [calculator, word_counter, lookup_product_price_safe], agent_prompt),
    tools=[calculator, word_counter, lookup_product_price_safe],
    verbose=True,
    handle_parsing_errors=True,
    handle_tool_error=True,   # catches ToolException and any tool-raised error, feeds it
                              # back to the model as an observation instead of raising
)


**What had to be configured, and how the agent recovers:** two things. First, the tool itself
does one manual retry — this is application logic LangChain doesn't provide for you. Second,
`AgentExecutor(handle_tool_error=True)` is what stops an uncaught tool exception from crashing the
whole `.invoke()` call: instead, the exception's string is inserted into the transcript as if it
were a normal tool observation, and the model gets to see "Error: ..." and decide what to do next
(usually: apologize, ask to retry, or fall back to a different tool). Without `handle_tool_error=True`,
a raised exception propagates and kills the run — which is actually closer to how Day 1's raw loop
behaved unless you wrapped every tool call in your own `try/except`.

### What LangChain made easier vs. what leaked through

LangChain made the boilerplate disappear: no more hand-rolling the JSON tool schema, the
reason/act/observe loop, or the message-history bookkeeping — `@tool`, `AgentExecutor`, and
`RunnableWithMessageHistory` cover all of that in a few lines each. But three things leaked through
this session: (1) **version churn** — the tutorial-standard `create_tool_calling_agent`/`AgentExecutor`
pattern is already the "classic," soon-to-be-legacy API in `langchain==1.x`, replaced by a
LangGraph-based `create_agent`, and `RunnableWithMessageHistory` is deprecated in favor of LangGraph
checkpointers; (2) **structured output doesn't compose cleanly with the agent loop** — getting a
Pydantic object out required a second chain rather than one flag on `AgentExecutor`; (3) **error
handling is still your job** — `handle_tool_error=True` only stops a crash, it doesn't add retries
or backoff, so the flaky-API resilience logic above is exactly the kind of code you'd have written
in Day 1's raw agent too.
